# 01 Bronze


In [ ]:
%pip install -q google-cloud-bigquery pandas pyarrow db-dtypes requests papermill
import os,sys,json,pandas as pd
from google.cloud import bigquery
PROJECT_ID="your-gcp-project-id"  # замініть у всіх ноутбуках
LOCATION="EU"; creds=None
if "google.colab" in sys.modules:
 from google.colab import auth; auth.authenticate_user()
elif os.environ.get("GCP_SA_KEY"):
 from google.oauth2 import service_account
 creds=service_account.Credentials.from_service_account_info(json.loads(os.environ["GCP_SA_KEY"]),scopes=["https://www.googleapis.com/auth/cloud-platform"])
client=bigquery.Client(project=PROJECT_ID,location=LOCATION,credentials=creds)
print("проєкт:",client.project)

In [ ]:
import requests,hashlib
for n in ["nbu_raw","nbu_dwh"]:
 d=bigquery.Dataset(f"{PROJECT_ID}.{n}"); d.location=LOCATION; client.create_dataset(d,exists_ok=True)
T=f"{PROJECT_ID}.nbu_raw.raw_rates"
s=[bigquery.SchemaField("ingested_at","TIMESTAMP","REQUIRED"),bigquery.SchemaField("business_date","DATE","REQUIRED"),bigquery.SchemaField("request_url","STRING","REQUIRED"),bigquery.SchemaField("payload","STRING","REQUIRED"),bigquery.SchemaField("payload_hash","STRING","REQUIRED")]
t=bigquery.Table(T,schema=s); t.time_partitioning=bigquery.TimePartitioning(field="business_date"); client.create_table(t,exists_ok=True)
URL="https://bank.gov.ua/NBUStatService/v1/statdirectory/exchange?json"; r=requests.get(URL,timeout=30); r.raise_for_status(); data=r.json(); print(len(data),"валют",data[0])
ts=pd.Timestamp.now(tz="UTC"); rows=[]
for x in data:
 p=json.dumps(x,ensure_ascii=False,sort_keys=True); rows.append({"ingested_at":ts,"business_date":pd.to_datetime(x["exchangedate"],format="%d.%m.%Y").date(),"request_url":URL,"payload":p,"payload_hash":hashlib.md5(p.encode()).hexdigest()})
df=pd.DataFrame(rows); display(df.head())
cfg=bigquery.LoadJobConfig(schema=s,write_disposition="WRITE_APPEND"); client.load_table_from_dataframe(df,T,job_config=cfg).result()
display(client.query(f"SELECT business_date,COUNT(*) rows_cnt,COUNT(DISTINCT payload_hash) uniq_cnt FROM `{T}` GROUP BY 1 ORDER BY 1").to_dataframe())
# rows_cnt подвоюється після повторного запуску через append; uniq_cnt незмінний, бо payload однаковий.